# 01 — Data Understanding

Goal of this notebook: understand the *structure* of the data before touching any
modeling or cleaning. Specifically we need to confirm, with numbers, three things
that drive every later decision:

1. How is the wide table laid out (band/month columns)?
2. Is train really "clean" (no missingness) while test is masked?
3. What exactly does the masking in test look like (window sizes, S1 vs S2 dropout)?

Everything here is read-only exploration — no cleaning, no feature engineering yet.

In [1]:
import sys
sys.path.append('..')  # so we can import from src/

import pandas as pd
import numpy as np

from src.config import ALL_BANDS, S1_BANDS, S2_BANDS, MONTHS, MISSING_VALUE
from src.data_utils import (
    load_raw_train, load_raw_test,
    month_missing_matrix, per_band_missing_rate,
    active_window_size, s2_dropout_given_s1_rate,
)

pd.set_option('display.max_columns', 20)

## 1. Load and inspect shape / columns

In [2]:
train = load_raw_train()
test = load_raw_test()

print("train shape:", train.shape)
print("test shape:", test.shape)
print()
print("train columns (first 16):", list(train.columns)[:16])
print()
print("Is train wide format band_month confirmed?",
      all(f"{b}_01" in train.columns for b in ALL_BANDS))

train shape: (1821, 146)
test shape: (1030, 145)

train columns (first 16): ['ID', 'label', 'VH_01', 'VV_01', 'blue_01', 'green_01', 'nir_01', 'nira_01', 're1_01', 're2_01', 're3_01', 'red_01', 'swir1_01', 'swir2_01', 'VH_02', 'VV_02']

Is train wide format band_month confirmed? True


**Why this matters:** confirms the wide format is `<band>_<month>` (e.g. `VH_01`),
12 bands x 12 months = 144 feature columns, plus `ID` and `label` in train
(test has `ID` only, no label — that's what we're predicting).

## 2. Label balance (train only)

In [3]:
print(train['label'].value_counts())
print()
print(train['label'].value_counts(normalize=True))

label
0    1086
1     735
Name: count, dtype: int64

label
0    0.596376
1    0.403624
Name: proportion, dtype: float64


**Why this matters:** confirms the ~40% positive rate stated in the competition
brief. This is mild imbalance, not extreme — matters for how we weight/evaluate,
but doesn't require heroic resampling.

## 3. Missingness: is train really fully populated?

In [4]:
train_missing_rate = per_band_missing_rate(train)
test_missing_rate = per_band_missing_rate(test)

missing_compare = pd.DataFrame({
    'train_missing_rate': train_missing_rate,
    'test_missing_rate': test_missing_rate,
})
missing_compare

,train_missing_rate,test_missing_rate
VH,0.0,0.583576
VV,0.0,0.583576
blue,0.0,0.609466
green,0.0,0.609466
nir,0.0,0.609466
nira,0.0,0.609466
re1,0.0,0.609466
re2,0.0,0.609466
re3,0.0,0.609466
red,0.0,0.609466


**Why this matters:** if `train_missing_rate` is ~0 for every band and
`test_missing_rate` is well over 50%, that confirms the central challenge of this
competition: train is clean, test is heavily masked. Our model will never see
partial data unless we manufacture it ourselves (notebook 03).

## 4. Test window sizes — how many months does each test row actually have?

In [5]:
test_window = active_window_size(test, band='VH')
print(pd.Series(test_window).value_counts().sort_index())
print()
print("train window sizes (sanity check, should all be 12):")
train_window = active_window_size(train, band='VH')
print(pd.Series(train_window).value_counts().sort_index())

4    345
5    343
6    342
Name: count, dtype: int64

train window sizes (sanity check, should all be 12):
12    1821
Name: count, dtype: int64


**Why this matters:** this tells us exactly what window sizes to replicate in our
masking-augmentation step. If test rows only ever have 4, 5, or 6 active months,
that's what we simulate on train — not some arbitrary random dropout rate.

## 5. Within an active month, does S2 (optical) still get cloud-masked?

In [6]:
s2_dropout_rate = s2_dropout_given_s1_rate(test, s2_band='blue')
print(f"Fraction of (row, month) pairs where S1 present but S2 (blue) missing: {s2_dropout_rate:.4f}")

# sanity check on train (should be ~0 since train has no missingness at all)
s2_dropout_rate_train = s2_dropout_given_s1_rate(train, s2_band='blue')
print(f"Same check on train (sanity, expect ~0): {s2_dropout_rate_train:.4f}")

Fraction of (row, month) pairs where S1 present but S2 (blue) missing: 0.0622
Same check on train (sanity, expect ~0): 0.0000


**Why this matters:** this is the secondary masking effect — even inside a test
row's "active window," individual optical months can still be cloud-dropped while
radar isn't. We'll replicate this rate during augmentation too, not just the
window-size masking.

## Summary of findings (fill in after running)

- Wide format confirmed: `<band>_<month>`, 12 bands x 12 months.
- Train label balance: ~40% positive (matches competition brief).
- Train missingness: effectively 0% across all bands — a fully clean dataset.
- Test missingness: 55-65%+ depending on band, because only a 4-6 month
  consecutive window survives per row.
- Test window sizes: roughly evenly split across 4, 5, and 6 months.
- Within an active window, S2 bands are still cloud-dropped in a small fraction
  of (row, month) pairs even though S1 is present — a secondary masking effect
  on top of the window itself.

**Implication carried into notebook 03:** we can't train on train as-is and expect
it to generalize to test's masking pattern. We need to manufacture masked copies
of train that match both of these measured rates (window size distribution +
S2-given-S1 dropout rate) before doing any feature engineering or modeling.